# OnomaCap J-text — Seoul pronounced-form transfer

Seoul Corpus의 **한글 `pWord.prono.`**를 utterance 경계 안에서 연속 chunk로 묶고 canonical Jamo로 분해해 BART를 denoising 사전학습한 뒤 OnomaCap에 전이합니다. chunk 경계와 pWord 시간 정렬은 versioned manifest에 고정해 이후 J-text·FLAC·alignment 실험이 동일한 샘플을 재사용합니다. 이 단계에서는 Seoul FLAC와 `phoneme` tier를 사용하지 않습니다. 예를 들어 phoneme tier의 `k0 a k7`을 토큰으로 매핑하지 않고, 발음형 `각`을 Unicode NFD로 분해한 `ᄀ ᅡ ᆨ`만 현재 OnomaCap Jamo 토큰으로 사용합니다. OnomaCap 전이 중 HTSAT는 완전히 동결합니다.


In [ ]:
# 의존성 설치
%pip install -q transformers==4.36.2 tokenizers==0.15.2 "huggingface-hub<1.0" torchlibrosa==0.1.0 librosa==0.10.2.post1 ruamel.yaml==0.17.40 gdown==5.2.0 "kagglehub>=0.3.12" pycocoevalcap==1.2 "scikit-learn>=1.4,<1.7" "pandas>=2.1,<2.4" matplotlib seaborn tqdm loguru warmup-scheduler gensim

# 저장소 clone 및 WavCaps overlay 적용
import shutil
from pathlib import Path

ONOMAHOW_REPO = 'https://github.com/youhan200203/OnomaHoW.git'
ONOMAHOW_REF = 'working'
WAVCAPS_REPO = 'https://github.com/XinhaoMei/WavCaps.git'
WAVCAPS_COMMIT = 'a5a9649ce305d7fe82cfcf5d6a4a12f03df9ef1e'
ANNOTATION_REPO = 'https://github.com/jspirit01/sound-to-onomatopoeia.git'

ONOMAHOW_DIR = Path('/content/OnomaHoW')
WAVCAPS_DIR = Path('/content/WavCaps')
ANNOTATION_DIR = Path('/content/sound-to-onomatopoeia')
for transient_dir in (ONOMAHOW_DIR, WAVCAPS_DIR, ANNOTATION_DIR):
    if transient_dir.exists():
        shutil.rmtree(transient_dir)

!git clone -q --depth 1 --branch "{ONOMAHOW_REF}" "{ONOMAHOW_REPO}" "{ONOMAHOW_DIR}"
!git clone -q "{WAVCAPS_REPO}" "{WAVCAPS_DIR}"
!git -C "{WAVCAPS_DIR}" checkout --detach -q "{WAVCAPS_COMMIT}"
!git clone -q --depth 1 "{ANNOTATION_REPO}" "{ANNOTATION_DIR}"
assert (ONOMAHOW_DIR / '.git').is_dir()
assert (WAVCAPS_DIR / '.git').is_dir()
assert (ANNOTATION_DIR / '.git').is_dir()

overlay_root = ONOMAHOW_DIR / 'wavcaps_patch'
overlay_files = sorted(overlay_root.rglob('*.py'))
assert overlay_files, 'wavcaps_patch overlay가 비어 있습니다.'
for source in overlay_files:
    destination = WAVCAPS_DIR / source.relative_to(overlay_root)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    assert source.read_bytes() == destination.read_bytes()

helper_destination = WAVCAPS_DIR / 'captioning/tools/jamo_preprocessing.py'
config_destination = WAVCAPS_DIR / 'captioning/settings/onomacap_jamo.yaml'
shutil.copy2(ONOMAHOW_DIR / 'jamo_preprocessing.py', helper_destination)
shutil.copy2(ONOMAHOW_DIR / 'configs/onomacap_jamo.yaml', config_destination)
checked_out_commit_output = !git -C "{WAVCAPS_DIR}" rev-parse HEAD
assert len(checked_out_commit_output) == 1, checked_out_commit_output
checked_out_commit = checked_out_commit_output[0].strip()
assert checked_out_commit == WAVCAPS_COMMIT
print(f'WavCaps {checked_out_commit} + {len(overlay_files)} overlay files')


In [ ]:
# 공통 설정과 canonical Jamo 정의
import csv
import gc
import hashlib
import json
import math
import os
import random
import re
import sys
import tarfile
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from google.colab import drive

MODEL_SEED = 20
SPLIT_SEED = 20
EVAL_BEAM_SIZE = 3
assert EVAL_BEAM_SIZE == 3
MAX_LENGTH = 80
CHUNK_MANIFEST_VERSION = 1
CHUNK_ALGORITHM = 'utterance_aligned_onomacap_train_lengths_v1'
CHUNK_MAX_JAMO = 55
EXPERIMENT_NAME = 'onomacap_jamo_seoul_jtext_chunk_v1'
CHOSEONG = tuple(chr(codepoint) for codepoint in range(0x1100, 0x1113))
JUNGSEONG = tuple(chr(codepoint) for codepoint in range(0x1161, 0x1176))
JONGSEONG = tuple(chr(codepoint) for codepoint in range(0x11A8, 0x11C3))
JAMO_VOCAB = CHOSEONG + JUNGSEONG + JONGSEONG
JAMO_SET = frozenset(JAMO_VOCAB)
assert len(JAMO_VOCAB) == 67

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(MODEL_SEED)
drive.mount('/content/drive')
SEOUL_DRIVE_ROOT = Path('/content/drive/MyDrive/OnomaCap/Seoul-Corpus')
JTEXT_DRIVE_ROOT = Path('/content/drive/MyDrive/OnomaCap/seoul_jtext')
SEOUL_PROCESSED_ROOT = SEOUL_DRIVE_ROOT / 'processed'
CHUNK_MANIFEST_PATH = SEOUL_PROCESSED_ROOT / 'seoul_chunk_manifest_v1.jsonl'
CHUNK_METADATA_PATH = SEOUL_PROCESSED_ROOT / 'seoul_chunk_metadata_v1.json'
ONOMA_SPLIT_MANIFEST = Path('/content/drive/MyDrive/OnomaCap/splits/onomacap_7961_seed20_v1.csv')
JTEXT_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
assert (SEOUL_DRIVE_ROOT / 'label.tgz').is_file()


In [ ]:
# label.tgz -> label-TextGrid.zip -> TextGrid (sound.tgz는 J-text에서 사용하지 않음)
SEOUL_WORK_ROOT = Path('/content/seoul_jtext_data')
if SEOUL_WORK_ROOT.exists():
    shutil.rmtree(SEOUL_WORK_ROOT)
OUTER_DIR = SEOUL_WORK_ROOT / 'outer'
TEXTGRID_DIR = SEOUL_WORK_ROOT / 'textgrids'
OUTER_DIR.mkdir(parents=True)
TEXTGRID_DIR.mkdir(parents=True)

def safe_destination(root, member_name):
    root = root.resolve()
    destination = (root / member_name).resolve()
    if destination != root and root not in destination.parents:
        raise ValueError(f'Unsafe archive member: {member_name!r}')
    return destination

with tarfile.open(SEOUL_DRIVE_ROOT / 'label.tgz', 'r:*') as archive:
    for member in archive.getmembers():
        safe_destination(OUTER_DIR, member.name)
    archive.extractall(OUTER_DIR)

nested_zips = sorted(OUTER_DIR.rglob('label-TextGrid.zip'))
if len(nested_zips) != 1:
    nested_zips = sorted(OUTER_DIR.rglob('*TextGrid*.zip'))
assert len(nested_zips) == 1, nested_zips
with zipfile.ZipFile(nested_zips[0]) as archive:
    for member in archive.infolist():
        safe_destination(TEXTGRID_DIR, member.filename)
    archive.extractall(TEXTGRID_DIR)

textgrid_paths = sorted(
    path for path in TEXTGRID_DIR.rglob('*.TextGrid')
    if '__MACOSX' not in path.parts and not path.name.startswith('._')
)
assert textgrid_paths, 'TextGrid가 없습니다.'
assert not list(SEOUL_WORK_ROOT.rglob('*.flac')), 'J-text 경로에 FLAC가 섞였습니다.'
print('nested zip:', nested_zips[0])
print('TextGrid files:', len(textgrid_paths))


In [ ]:
# UTF-16 TextGrid ordered-tier parser: 중복 tier 이름을 보존한다.
EXPECTED_TIER_NAMES = (
    'phoneme',
    'pWord.prono.',
    'pWord.prono.',
    'utt.prono.',
    'pWord.ortho.',
    'pWord.ortho.',
    'utt.ortho.',
)
ITEM_RE = re.compile(r'(?m)^\s*item \[(\d+)\]:\s*$')
NAME_RE = re.compile(r'(?m)^\s*name = "([^"]*)"\s*$')
INTERVAL_RE = re.compile(
    r'intervals \[(\d+)\]:\s*'
    r'xmin = ([0-9.eE+-]+)\s*'
    r'xmax = ([0-9.eE+-]+)\s*'
    r'text = "([^"]*)"',
    re.S,
)
HANGUL_WORD_RE = re.compile(r'^[가-힣 ]+$')
EVENT_RE = re.compile(r'^<[^>]+>$')
SPEAKER_FILE_RE = re.compile(r'^(?P<speaker>s\d+[fm]\d+)[fm]\d+$')

@dataclass(frozen=True)
class TextGridInterval:
    index: int
    xmin: float
    xmax: float
    text: str

@dataclass(frozen=True)
class TextGridTier:
    name: str
    intervals: tuple

def read_textgrid_ordered(path):
    raw_bytes = Path(path).read_bytes()
    if raw_bytes.startswith((b'\xfe\xff', b'\xff\xfe')):
        text = raw_bytes.decode('utf-16')
    else:
        text = raw_bytes.decode('utf-8-sig')
    matches = list(ITEM_RE.finditer(text))
    tiers = []
    for offset, match in enumerate(matches):
        end = matches[offset + 1].start() if offset + 1 < len(matches) else len(text)
        block = text[match.end():end]
        name_match = NAME_RE.search(block)
        if name_match is None:
            raise ValueError(f'Missing tier name: {path}, item={match.group(1)}')
        intervals = tuple(
            TextGridInterval(int(index), float(xmin), float(xmax), value)
            for index, xmin, xmax, value in INTERVAL_RE.findall(block)
        )
        tiers.append(TextGridTier(name_match.group(1), intervals))
    names = tuple(tier.name for tier in tiers)
    if names != EXPECTED_TIER_NAMES:
        raise ValueError(f'Unexpected tier order in {path}: {names!r}')
    hangul_values = [item.text for item in tiers[1].intervals if HANGUL_WORD_RE.fullmatch(item.text)]
    roman_values = [item.text for item in tiers[2].intervals if not EVENT_RE.fullmatch(item.text)]
    if not hangul_values or any(re.search(r'[가-힣]', value) for value in roman_values):
        raise ValueError(f'Hangul/romanized pWord tiers are reversed or malformed: {path}')
    if len(tiers[1].intervals) != len(tiers[2].intervals):
        raise ValueError(f'Pronounced pWord tier length mismatch: {path}')
    return tuple(tiers)

def speaker_id_from_path(path):
    match = SPEAKER_FILE_RE.fullmatch(Path(path).stem)
    if match is None:
        raise ValueError(f'Unknown Seoul filename: {Path(path).name}')
    return match.group('speaker')

def pronounced_hangul_to_jamo(text):
    normalized = unicodedata.normalize('NFC', str(text))
    compact = ''.join(normalized.split())
    if not compact or not all(0xAC00 <= ord(character) <= 0xD7A3 for character in compact):
        raise ValueError(f'Not a precomposed-Hangul pronounced form: {text!r}')
    decomposed = unicodedata.normalize('NFD', compact)
    if not decomposed or set(decomposed) - JAMO_SET:
        raise ValueError(f'Unsupported canonical Jamo in {text!r}: {decomposed!r}')
    reconstructed = unicodedata.normalize('NFC', decomposed)
    if reconstructed != compact:
        raise ValueError(f'Jamo round-trip failed: {text!r} -> {reconstructed!r}')
    return ' '.join(decomposed)

# phoneme 표기는 현재 자모 tokenizer와 다른 체계이며 J-text 입력으로 사용하지 않는다.
assert pronounced_hangul_to_jamo('각') == 'ᄀ ᅡ ᆨ'
assert set('k0 a k7'.split()).isdisjoint(JAMO_SET)
sample_tiers = read_textgrid_ordered(textgrid_paths[0])
print(tuple(tier.name for tier in sample_tiers))
print('phoneme example (excluded):', [item.text for item in sample_tiers[0].intervals[:8]])
print('Hangul pWord.prono. examples:', [item.text for item in sample_tiers[1].intervals[:12]])


In [ ]:
# OnomaCap train 길이 분포와 Seoul utterance-aligned pWord run 준비
if str(ONOMAHOW_DIR) not in sys.path:
    sys.path.insert(0, str(ONOMAHOW_DIR))
from jamo_preprocessing import EXPECTED_LATIN_AUDIO, EXPECTED_OUTPUT_ROWS, load_or_create_split_manifest, prepare_rows

csv_path = ANNOTATION_DIR / 'sound-to-onomatopoeia_annotation.csv'
with csv_path.open(encoding='utf-8-sig', newline='') as stream:
    prepared_rows, onoma_report = prepare_rows(csv.DictReader(stream))
assert onoma_report.output_rows == EXPECTED_OUTPUT_ROWS == 7_961
assert onoma_report.latin_rows_excluded == (EXPECTED_LATIN_AUDIO,)
df = pd.DataFrame(prepared_rows)
jamo_columns = [f'candidate{index}_jamo' for index in range(1, 6)]
assert len(df) == 7_961 and df['class'].nunique() == 41
assert df[jamo_columns].map(lambda value: set(value.split()) <= JAMO_SET).all().all()
splits = load_or_create_split_manifest(df, ONOMA_SPLIT_MANIFEST, split_seed=SPLIT_SEED)
assert {name: len(frame) for name, frame in splits.items()} == {'train': 6_368, 'val': 796, 'test': 797}
onoma_train_lengths = np.asarray([
    len(str(row[column]).split())
    for _, row in splits['train'].iterrows()
    for column in jamo_columns
], dtype=np.int64)
assert len(onoma_train_lengths) == 31_840
assert int(onoma_train_lengths.max()) <= CHUNK_MAX_JAMO
assert CHUNK_MAX_JAMO + 2 <= MAX_LENGTH

def extract_word_runs(path):
    tiers = read_textgrid_ordered(path)
    speaker_id = speaker_id_from_path(path)
    utterances = tiers[3].intervals
    runs = []
    events = Counter()
    rejected = Counter()
    utterance_position = 0
    current_utterance = None
    current_words = []
    run_index = 0

    def flush_run():
        nonlocal current_words, run_index
        if current_words:
            runs.append({
                'source_textgrid': path.relative_to(TEXTGRID_DIR).as_posix(),
                'speaker_id': speaker_id,
                'utterance_index': current_utterance.index,
                'utterance_xmin': current_utterance.xmin,
                'utterance_xmax': current_utterance.xmax,
                'run_index': run_index,
                'words': current_words,
            })
            run_index += 1
            current_words = []

    # tier 2의 한글 pWord만 사용하며 tier 4의 utterance 시간 경계를 넘지 않는다.
    for interval in tiers[1].intervals:
        midpoint = (interval.xmin + interval.xmax) / 2
        while utterance_position < len(utterances) and midpoint > utterances[utterance_position].xmax + 1e-6:
            flush_run()
            utterance_position += 1
            current_utterance = None
            run_index = 0
        if utterance_position >= len(utterances) or midpoint < utterances[utterance_position].xmin - 1e-6:
            flush_run()
            rejected['outside_utterance'] += 1
            continue
        utterance = utterances[utterance_position]
        if current_utterance is None or current_utterance.index != utterance.index:
            flush_run()
            current_utterance = utterance
            run_index = 0
        value = interval.text.strip()
        if EVENT_RE.fullmatch(value):
            events[value] += 1
            flush_run()
            continue
        if not HANGUL_WORD_RE.fullmatch(value):
            rejected['non_hangul'] += 1
            flush_run()
            continue
        compact = ''.join(value.split())
        jamo = pronounced_hangul_to_jamo(compact)
        jamo_length = len(jamo.split())
        if jamo_length > CHUNK_MAX_JAMO:
            rejected['single_pword_over_chunk_max'] += 1
            flush_run()
            continue
        current_words.append({
            'interval_index': interval.index,
            'xmin': interval.xmin,
            'xmax': interval.xmax,
            'pronounced_hangul': compact,
            'canonical_jamo': jamo,
            'jamo_length': jamo_length,
        })
    flush_run()
    return runs, events, rejected

word_runs = []
event_counts = Counter()
rejected_counts = Counter()
for path in textgrid_paths:
    path_runs, path_events, path_rejected = extract_word_runs(path)
    word_runs.extend(path_runs)
    event_counts.update(path_events)
    rejected_counts.update(path_rejected)

assert word_runs
unexpected_rejections = {key: value for key, value in rejected_counts.items() if key != 'outside_utterance' and value}
assert not unexpected_rejections, unexpected_rejections
speaker_ids = sorted({run['speaker_id'] for run in word_runs})
pword_rows = [word for run in word_runs for word in run['words']]
pword_lengths = np.asarray([word['jamo_length'] for word in pword_rows])
source_report = {
    'textgrid_files': len(textgrid_paths),
    'speakers': len(speaker_ids),
    'accepted_pwords': len(pword_rows),
    'word_runs': len(word_runs),
    'events_excluded_total': sum(event_counts.values()),
    'pword_jamo_median': float(np.median(pword_lengths)),
    'onomacap_train_jamo_median': float(np.median(onoma_train_lengths)),
    'onomacap_train_jamo_p95': float(np.percentile(onoma_train_lengths, 95)),
    'onomacap_train_jamo_max': int(onoma_train_lengths.max()),
    'phoneme_tier_used_for_training': False,
    'romanized_pword_tier_used_for_training': False,
    'flac_used_for_training': False,
}
display(source_report)


In [ ]:
# 화자 split과 결정론적 chunk manifest 생성 또는 검증 후 재사용
SPEAKER_META_RE = re.compile(r'^s\d+(?P<gender>[mf])(?P<age>\d+)$')
strata = defaultdict(list)
for speaker_id in speaker_ids:
    match = SPEAKER_META_RE.fullmatch(speaker_id)
    if match is None:
        raise ValueError(f'Cannot parse Seoul speaker metadata: {speaker_id!r}')
    decade = int(match.group('age')) // 10
    strata[(match.group('gender'), decade)].append(speaker_id)

split_rng = random.Random(SPLIT_SEED)
train_speakers, val_speakers = set(), set()
for stratum, members in sorted(strata.items()):
    members = sorted(members)
    split_rng.shuffle(members)
    val_count = max(1, round(len(members) * 0.2))
    val_speakers.update(members[:val_count])
    train_speakers.update(members[val_count:])
assert train_speakers and val_speakers and train_speakers.isdisjoint(val_speakers)

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()

def write_jsonl(path, rows):
    with Path(path).open('w', encoding='utf-8') as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n')

def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as stream:
        return [json.loads(line) for line in stream if line.strip()]

target_length_values = tuple(int(value) for value in onoma_train_lengths.tolist())
target_length_fingerprint = hashlib.sha256(
    json.dumps(target_length_values, separators=(',', ':')).encode('utf-8')
).hexdigest()
label_archive_fingerprint = sha256_file(SEOUL_DRIVE_ROOT / 'label.tgz')
onoma_split_fingerprint = sha256_file(ONOMA_SPLIT_MANIFEST)
target_tail_threshold = int(np.percentile(onoma_train_lengths, 5))

def segment_word_run(run):
    seed_key = f"{SPLIT_SEED}:{run['source_textgrid']}:{run['utterance_index']}:{run['run_index']}"
    stable_seed = int(hashlib.sha256(seed_key.encode('utf-8')).hexdigest()[:16], 16)
    rng = random.Random(stable_seed)
    words = run['words']
    segments = []
    position = 0
    while position < len(words):
        target_length = rng.choice(target_length_values)
        segment_words = [words[position]]
        segment_length = words[position]['jamo_length']
        position += 1
        while position < len(words):
            candidate_length = segment_length + words[position]['jamo_length']
            if candidate_length > CHUNK_MAX_JAMO:
                break
            if abs(candidate_length - target_length) > abs(segment_length - target_length):
                break
            segment_words.append(words[position])
            segment_length = candidate_length
            position += 1
        segments.append({
            'words': segment_words,
            'target_jamo_length': target_length,
            'tail_merged': False,
        })
    if len(segments) >= 2:
        tail = segments[-1]
        previous = segments[-2]
        tail_length = sum(word['jamo_length'] for word in tail['words'])
        merged_length = tail_length + sum(word['jamo_length'] for word in previous['words'])
        if tail_length < target_tail_threshold and merged_length <= CHUNK_MAX_JAMO:
            previous['words'].extend(tail['words'])
            previous['tail_merged'] = True
            segments.pop()

    chunk_rows = []
    for chunk_index, segment in enumerate(segments):
        segment_words = segment['words']
        interval_indices = [word['interval_index'] for word in segment_words]
        identity = {
            'version': CHUNK_MANIFEST_VERSION,
            'source_textgrid': run['source_textgrid'],
            'utterance_index': run['utterance_index'],
            'pword_interval_indices': interval_indices,
        }
        chunk_id = hashlib.sha256(
            json.dumps(identity, sort_keys=True, separators=(',', ':')).encode('utf-8')
        ).hexdigest()
        pronounced_pwords = [word['pronounced_hangul'] for word in segment_words]
        canonical_jamo = ' '.join(word['canonical_jamo'] for word in segment_words)
        chunk_rows.append({
            'chunk_id': chunk_id,
            'split': 'train' if run['speaker_id'] in train_speakers else 'val',
            'speaker_id': run['speaker_id'],
            'source_textgrid': run['source_textgrid'],
            'source_stem': Path(run['source_textgrid']).stem,
            'utterance_index': run['utterance_index'],
            'utterance_xmin': run['utterance_xmin'],
            'utterance_xmax': run['utterance_xmax'],
            'run_index': run['run_index'],
            'chunk_index': chunk_index,
            'chunk_xmin': segment_words[0]['xmin'],
            'chunk_xmax': segment_words[-1]['xmax'],
            'pword_interval_indices': interval_indices,
            'pword_xmins': [word['xmin'] for word in segment_words],
            'pword_xmaxs': [word['xmax'] for word in segment_words],
            'pronounced_pwords': pronounced_pwords,
            'pronounced_hangul': ' '.join(pronounced_pwords),
            'canonical_jamo': canonical_jamo,
            'jamo_length': len(canonical_jamo.split()),
            'target_jamo_length': segment['target_jamo_length'],
            'tail_merged': segment['tail_merged'],
        })
    return chunk_rows

expected_metadata = {
    'version': CHUNK_MANIFEST_VERSION,
    'algorithm': CHUNK_ALGORITHM,
    'split_seed': SPLIT_SEED,
    'segmentation_seed': SPLIT_SEED,
    'chunk_max_jamo': CHUNK_MAX_JAMO,
    'event_policy': 'exclude_entire_angle_bracket_interval_and_break_run',
    'pword_tier_number': 2,
    'utterance_tier_number': 4,
    'phoneme_used': False,
    'flac_used': False,
    'textgrid_file_count': len(textgrid_paths),
    'source_label_tgz_sha256': label_archive_fingerprint,
    'onomacap_split_manifest_sha256': onoma_split_fingerprint,
    'onomacap_train_length_sha256': target_length_fingerprint,
    'onomacap_train_length_count': len(target_length_values),
    'train_speakers': sorted(train_speakers),
    'val_speakers': sorted(val_speakers),
}
manifest_exists = CHUNK_MANIFEST_PATH.is_file()
metadata_exists = CHUNK_METADATA_PATH.is_file()
if manifest_exists != metadata_exists:
    raise RuntimeError('Chunk dataset is incomplete: manifest and metadata must both exist or both be absent.')

if not manifest_exists:
    chunk_records = [chunk for run in word_runs for chunk in segment_word_run(run)]
    chunk_records.sort(key=lambda row: (row['source_textgrid'], row['utterance_index'], row['run_index'], row['chunk_index']))
    CHUNK_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    manifest_temporary = CHUNK_MANIFEST_PATH.with_suffix(CHUNK_MANIFEST_PATH.suffix + '.tmp')
    metadata_temporary = CHUNK_METADATA_PATH.with_suffix(CHUNK_METADATA_PATH.suffix + '.tmp')
    write_jsonl(manifest_temporary, chunk_records)
    chunk_lengths = np.asarray([row['jamo_length'] for row in chunk_records])
    metadata = {
        **expected_metadata,
        'manifest_sha256': sha256_file(manifest_temporary),
        'chunk_count': len(chunk_records),
        'split_counts': dict(Counter(row['split'] for row in chunk_records)),
        'chunk_jamo_median': float(np.median(chunk_lengths)),
        'chunk_jamo_p95': float(np.percentile(chunk_lengths, 95)),
        'chunk_jamo_max': int(chunk_lengths.max()),
        'source_report': source_report,
    }
    metadata_temporary.write_text(json.dumps(metadata, ensure_ascii=False, indent=2, sort_keys=True), encoding='utf-8')
    os.replace(manifest_temporary, CHUNK_MANIFEST_PATH)
    os.replace(metadata_temporary, CHUNK_METADATA_PATH)
    print('Created immutable chunk dataset:', CHUNK_MANIFEST_PATH)
else:
    metadata = json.loads(CHUNK_METADATA_PATH.read_text(encoding='utf-8'))
    mismatches = {key: (metadata.get(key), value) for key, value in expected_metadata.items() if metadata.get(key) != value}
    if mismatches:
        raise ValueError(f'Chunk metadata mismatch; create a new manifest version instead of overwriting v1: {mismatches}')
    if metadata.get('manifest_sha256') != sha256_file(CHUNK_MANIFEST_PATH):
        raise ValueError('Chunk manifest checksum mismatch.')
    chunk_records = read_jsonl(CHUNK_MANIFEST_PATH)
    if int(metadata.get('chunk_count', -1)) != len(chunk_records):
        raise ValueError('Chunk manifest row count mismatch.')
    print('Loaded immutable chunk dataset:', CHUNK_MANIFEST_PATH)

assert chunk_records
assert len({row['chunk_id'] for row in chunk_records}) == len(chunk_records)
assert {row['split'] for row in chunk_records} == {'train', 'val'}
assert dict(Counter(row['split'] for row in chunk_records)) == metadata['split_counts']
assert all(row['speaker_id'] in (train_speakers if row['split'] == 'train' else val_speakers) for row in chunk_records)
assert all(0 < row['jamo_length'] <= CHUNK_MAX_JAMO for row in chunk_records)
assert all(row['jamo_length'] == len(row['canonical_jamo'].split()) for row in chunk_records)
assert all(set(row['canonical_jamo'].split()) <= JAMO_SET for row in chunk_records)
assert all(len(row['pword_interval_indices']) == len(row['pword_xmins']) == len(row['pword_xmaxs']) == len(row['pronounced_pwords']) for row in chunk_records)
assert all(row['pword_interval_indices'] == sorted(row['pword_interval_indices']) for row in chunk_records)
assert all(row['pword_xmins'] == sorted(row['pword_xmins']) and row['pword_xmaxs'] == sorted(row['pword_xmaxs']) for row in chunk_records)
assert all(row['chunk_xmin'] == row['pword_xmins'][0] and row['chunk_xmax'] == row['pword_xmaxs'][-1] for row in chunk_records)
assert all(row['utterance_xmin'] - 1e-6 <= row['chunk_xmin'] < row['chunk_xmax'] <= row['utterance_xmax'] + 1e-6 for row in chunk_records)
assert all(row['pronounced_hangul'] == ' '.join(row['pronounced_pwords']) for row in chunk_records)
assert all(
    unicodedata.normalize('NFC', ''.join(row['canonical_jamo'].split())) == ''.join(row['pronounced_hangul'].split())
    for row in chunk_records
)
train_records = [row for row in chunk_records if row['split'] == 'train']
val_records = [row for row in chunk_records if row['split'] == 'val']
chunk_lengths = np.asarray([row['jamo_length'] for row in chunk_records])
processing_report = {
    'chunks': len(chunk_records),
    'train_chunks': len(train_records),
    'val_chunks': len(val_records),
    'chunk_jamo_median': float(np.median(chunk_lengths)),
    'chunk_jamo_p95': float(np.percentile(chunk_lengths, 95)),
    'chunk_jamo_max': int(chunk_lengths.max()),
    'onomacap_train_jamo_median': float(np.median(onoma_train_lengths)),
    'onomacap_train_jamo_p95': float(np.percentile(onoma_train_lengths, 95)),
    'manifest': str(CHUNK_MANIFEST_PATH),
    'metadata': str(CHUNK_METADATA_PATH),
}
display(processing_report)
display(pd.DataFrame(chunk_records)[['chunk_id', 'split', 'source_textgrid', 'speaker_id', 'chunk_xmin', 'chunk_xmax', 'pronounced_hangul', 'canonical_jamo', 'jamo_length']].head())


In [ ]:
# J-text BART와 음절 단위 denoising batch 구성
from torch.utils.data import DataLoader, Dataset
from transformers import BartConfig, BartForConditionalGeneration, BartTokenizer

TOKENIZER_NAME = 'facebook/bart-base'
tokenizer = BartTokenizer.from_pretrained(TOKENIZER_NAME)
base_vocab_size = tokenizer.vocab_size
added_count = tokenizer.add_tokens(list(JAMO_VOCAB))
assert added_count == len(JAMO_VOCAB)
jamo_to_id = {token: tokenizer.convert_tokens_to_ids(token) for token in JAMO_VOCAB}
assert sorted(jamo_to_id.values()) == list(range(base_vocab_size, base_vocab_size + 67))
id_to_jamo = {token_id: token for token, token_id in jamo_to_id.items()}

bart_config = BartConfig.from_pretrained(TOKENIZER_NAME)
jtext_model = BartForConditionalGeneration(bart_config)
jtext_model.resize_token_embeddings(len(tokenizer))
assert jtext_model.config.vocab_size == len(tokenizer)

class SeoulJTextDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        return self.rows[index]

class JamoDenoisingCollator:
    def __init__(self, training, seed):
        self.training = bool(training)
        self.seed = int(seed)

    def _rng(self, row):
        if self.training:
            return random
        key = f"{self.seed}:{row['chunk_id']}"
        stable_seed = int(hashlib.sha256(key.encode()).hexdigest()[:16], 16)
        return random.Random(stable_seed)

    def __call__(self, rows):
        sources, targets = [], []
        for row in rows:
            compact_hangul = ''.join(row['pronounced_hangul'].split())
            syllables = [list(unicodedata.normalize('NFD', syllable)) for syllable in compact_hangul]
            rng = self._rng(row)
            mask_count = max(1, round(len(syllables) * 0.3))
            masked = set(rng.sample(range(len(syllables)), k=min(mask_count, len(syllables))))
            source_ids = [tokenizer.bos_token_id]
            target_ids = [tokenizer.bos_token_id]
            for index, components in enumerate(syllables):
                component_ids = [jamo_to_id[token] for token in components]
                source_ids.extend([tokenizer.mask_token_id] if index in masked else component_ids)
                target_ids.extend(component_ids)
            source_ids.append(tokenizer.eos_token_id)
            target_ids.append(tokenizer.eos_token_id)
            assert len(source_ids) <= MAX_LENGTH and len(target_ids) <= MAX_LENGTH
            sources.append(source_ids)
            targets.append(target_ids)

        source_length = max(map(len, sources))
        target_length = max(map(len, targets))
        input_ids = torch.full((len(rows), source_length), tokenizer.pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros_like(input_ids)
        labels = torch.full((len(rows), target_length), -100, dtype=torch.long)
        for row_index, (source_ids, target_ids) in enumerate(zip(sources, targets)):
            input_ids[row_index, :len(source_ids)] = torch.tensor(source_ids)
            attention_mask[row_index, :len(source_ids)] = 1
            labels[row_index, :len(target_ids)] = torch.tensor(target_ids)
        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

train_dataset = SeoulJTextDataset(train_records)
val_dataset = SeoulJTextDataset(val_records)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0, collate_fn=JamoDenoisingCollator(True, MODEL_SEED))
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0, collate_fn=JamoDenoisingCollator(False, MODEL_SEED))
batch = next(iter(val_loader))
assert set(batch) == {'input_ids', 'attention_mask', 'labels'}
print('BART base vocabulary:', base_vocab_size)
print('BART + Jamo vocabulary:', len(tokenizer))
print({name: tuple(value.shape) for name, value in batch.items()})


In [ ]:
# J-text 학습, validation loss, 최종 평가용 edit distance
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import get_cosine_schedule_with_warmup
from transformers.models.bart.modeling_bart import shift_tokens_right

def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for ref_token in reference:
        current = [previous[0] + 1]
        for column, hyp_token in enumerate(hypothesis, start=1):
            current.append(min(current[-1] + 1, previous[column] + 1, previous[column - 1] + (ref_token != hyp_token)))
        previous = current
    return previous[-1]

def forward_loss(model, batch, device):
    batch = {name: value.to(device, non_blocking=True) for name, value in batch.items()}
    decoder_input_ids = shift_tokens_right(batch['labels'], model.config.pad_token_id, model.config.decoder_start_token_id)
    outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], decoder_input_ids=decoder_input_ids, labels=None, return_dict=True)
    loss = F.cross_entropy(outputs.logits.reshape(-1, outputs.logits.size(-1)), batch['labels'].reshape(-1), ignore_index=-100, label_smoothing=0.1)
    return loss

def train_one_epoch(model, loader, optimizer, scheduler, device, epoch, accumulation_steps):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    progress = tqdm(loader, desc=f'J-text train epoch {epoch}', unit='batch')
    for batch_index, batch in enumerate(progress, start=1):
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = forward_loss(model, batch, device)
        if not torch.isfinite(loss):
            raise FloatingPointError(f'Non-finite J-text loss: epoch={epoch}, batch={batch_index}')
        (loss / accumulation_steps).backward()
        if batch_index % accumulation_steps == 0 or batch_index == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0, error_if_nonfinite=True)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        total_loss += float(loss.detach().cpu())
        progress.set_postfix(loss=f'{total_loss / batch_index:.4f}', lr=f"{scheduler.get_last_lr()[0]:.2e}")
    return total_loss / len(loader)

@torch.no_grad()
def validate_loss(model, loader, device):
    model.eval()
    total_loss = 0.0
    for batch in tqdm(loader, desc='J-text validation loss', unit='batch'):
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            total_loss += float(forward_loss(model, batch, device).cpu())
    return total_loss / len(loader)


In [ ]:
# J-text 사전학습: tqdm이 batch 진행률과 ETA를 계속 표시한다.
assert torch.cuda.is_available(), 'CUDA GPU가 필요합니다.'
assert torch.cuda.is_bf16_supported(), 'BF16 지원 CUDA GPU가 필요합니다.'
device = 'cuda'
jtext_model = jtext_model.to(device)
JTEXT_EPOCHS = 20
ACCUMULATION_STEPS = 4
EARLY_STOPPING_PATIENCE = 3
BEST_JTEXT_PATH = JTEXT_DRIVE_ROOT / 'best_jtext_decoder_chunk_v1_val_loss.pt'
LAST_JTEXT_PATH = JTEXT_DRIVE_ROOT / 'last_jtext_training_chunk_v1_val_loss.pt'

optimizer = torch.optim.AdamW(jtext_model.parameters(), lr=1e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-6)
updates_per_epoch = math.ceil(len(train_loader) / ACCUMULATION_STEPS)
total_updates = updates_per_epoch * JTEXT_EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=max(1, round(total_updates * 0.05)), num_training_steps=total_updates)
start_epoch, best_val_loss, stale_epochs = 1, float('inf'), 0
history = []
if LAST_JTEXT_PATH.is_file():
    resume = torch.load(LAST_JTEXT_PATH, map_location=device, weights_only=False)
    if resume.get('chunk_manifest_sha256') != metadata['manifest_sha256']:
        raise ValueError('J-text resume checkpoint was trained with a different chunk manifest.')
    jtext_model.load_state_dict(resume['decoder'])
    optimizer.load_state_dict(resume['optimizer'])
    scheduler.load_state_dict(resume['scheduler'])
    start_epoch = int(resume['epoch']) + 1
    if resume.get('selection_metric') != 'val_loss':
        raise ValueError('J-text resume checkpoint does not use validation loss selection.')
    best_val_loss = float(resume['best_val_loss'])
    stale_epochs = int(resume['stale_epochs'])
    history = list(resume['history'])
    print('Resume J-text at epoch', start_epoch)

for epoch in range(start_epoch, JTEXT_EPOCHS + 1):
    epoch_started = time.time()
    train_loss = train_one_epoch(jtext_model, train_loader, optimizer, scheduler, device, epoch, ACCUMULATION_STEPS)
    val_loss = validate_loss(jtext_model, val_loader, device)
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'minutes': (time.time() - epoch_started) / 60}
    history.append(row)
    print(row)
    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        stale_epochs = 0
        torch.save({'decoder': {name: value.detach().cpu() for name, value in jtext_model.state_dict().items()}, 'jamo_to_id': jamo_to_id, 'tokenizer_length': len(tokenizer), 'base_vocab_size': base_vocab_size, 'epoch': epoch, 'val_loss': val_loss, 'selection_metric': 'val_loss', 'config': {'model': TOKENIZER_NAME, 'max_length': MAX_LENGTH, 'source_tier_number': 2, 'source_tier_name': 'pWord.prono.', 'chunk_manifest_version': CHUNK_MANIFEST_VERSION, 'chunk_algorithm': CHUNK_ALGORITHM, 'phoneme_used': False, 'flac_used': False}, 'chunk_manifest': str(CHUNK_MANIFEST_PATH), 'chunk_metadata': str(CHUNK_METADATA_PATH), 'chunk_manifest_sha256': metadata['manifest_sha256']}, BEST_JTEXT_PATH)
    else:
        stale_epochs += 1
    torch.save({'decoder': jtext_model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(), 'epoch': epoch, 'best_val_loss': best_val_loss, 'selection_metric': 'val_loss', 'stale_epochs': stale_epochs, 'history': history, 'chunk_manifest': str(CHUNK_MANIFEST_PATH), 'chunk_manifest_sha256': metadata['manifest_sha256']}, LAST_JTEXT_PATH)
    if stale_epochs >= EARLY_STOPPING_PATIENCE:
        print(f'Early stopping at epoch {epoch}; best validation loss={best_val_loss:.6f}')
        break

assert BEST_JTEXT_PATH.is_file()
best_jtext = torch.load(BEST_JTEXT_PATH, map_location='cpu', weights_only=False)
assert best_jtext['chunk_manifest_sha256'] == metadata['manifest_sha256']
assert best_jtext['selection_metric'] == 'val_loss'
assert best_jtext['config']['phoneme_used'] is False
assert best_jtext['config']['flac_used'] is False
display(pd.DataFrame(history))
print('best J-text:', BEST_JTEXT_PATH, 'epoch', best_jtext['epoch'], 'validation loss', best_jtext['val_loss'])
del jtext_model, optimizer, scheduler
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# OnomaCap canonical-Jamo 데이터 준비
import kagglehub
audio_root = Path(kagglehub.dataset_download('buraktaci/firat-esc50'))
assert len(df) == 7_961 and {name: len(frame) for name, frame in splits.items()} == {'train': 6_368, 'val': 796, 'test': 797}

def audio_key(name):
    return unicodedata.normalize('NFKC', Path(str(name)).name).strip().casefold()

extensions = {'.mp3', '.wav', '.flac', '.ogg', '.m4a'}
audio_files = {audio_key(path.name): path for path in audio_root.rglob('*') if path.suffix.lower() in extensions}
df['audio_path'] = df['audio_file'].map(lambda name: str(audio_files.get(audio_key(name), '')))
assert not df['audio_path'].eq('').any()
assert df[jamo_columns].map(lambda value: set(value.split()) <= JAMO_SET).all().all()
splits = load_or_create_split_manifest(df, ONOMA_SPLIT_MANIFEST, split_seed=SPLIT_SEED)

JSON_DIR = WAVCAPS_DIR / 'captioning/data/OnomaCap/json_files'
JSON_DIR.mkdir(parents=True, exist_ok=True)
for split_name, frame in splits.items():
    data = []
    for _, row in frame.iterrows():
        data.append({'audio': str(Path(row['audio_path']).resolve()), **{f'caption_{index}': str(row[f'candidate{index}_jamo']) for index in range(1, 6)}})
    (JSON_DIR / f'{split_name}.json').write_text(json.dumps({'data': data}, ensure_ascii=False, indent=2), encoding='utf-8')
    print(split_name, len(data))

HTSAT_SOURCE = Path('/content/drive/MyDrive/OnomaCap/pretrained/HTSAT.ckpt')
HTSAT_DESTINATION = WAVCAPS_DIR / 'captioning/pretrained_models/audio_encoder/HTSAT.ckpt'
if not HTSAT_SOURCE.is_file():
    raise FileNotFoundError(f'{HTSAT_SOURCE}가 없습니다.')
HTSAT_DESTINATION.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(HTSAT_SOURCE, HTSAT_DESTINATION)
assert HTSAT_DESTINATION.stat().st_size == HTSAT_SOURCE.stat().st_size


In [ ]:
# J-text transfer 설정과 strict frozen-HTSAT smoke test
import ruamel.yaml as yaml
jamo_yaml = yaml.YAML()
with config_destination.open('r', encoding='utf-8') as stream:
    transfer_config = jamo_yaml.load(stream)
transfer_config['exp_name'] = EXPERIMENT_NAME
transfer_config['seed'] = MODEL_SEED
transfer_config['pretrain'] = False
transfer_config.pop('pretrain_path', None)
transfer_config.pop('pretrain_strict', None)
transfer_config['text_pretrain_path'] = str(BEST_JTEXT_PATH)
transfer_config['audio_encoder_args']['freeze'] = True
transfer_config['audio_encoder_args']['spec_augment'] = False
transfer_config['text_decoder_args']['pretrained'] = False
transfer_config['text_decoder_args']['compact_jamo_vocab'] = False
transfer_config['evaluation']['beam_size'] = EVAL_BEAM_SIZE
with config_destination.open('w', encoding='utf-8') as stream:
    jamo_yaml.dump(transfer_config, stream)

captioning_dir = WAVCAPS_DIR / 'captioning'
previous_cwd = Path.cwd()
smoke_model = None
try:
    os.chdir(captioning_dir)
    if str(captioning_dir) not in sys.path:
        sys.path.insert(0, str(captioning_dir))
    from data_handling.datamodule import AudioCaptionDataModule
    from models.bart_captioning import BartCaptionModel
    smoke_data = AudioCaptionDataModule(transfer_config, 'OnomaCap')
    audio, text, _, _ = next(iter(smoke_data.train_dataloader()))
    smoke_model = BartCaptionModel(transfer_config).to('cuda')
    jtext_checkpoint = torch.load(BEST_JTEXT_PATH, map_location='cpu', weights_only=False)
    assert jtext_checkpoint['jamo_to_id'] == smoke_model.jamo_to_id
    assert jtext_checkpoint['tokenizer_length'] == len(smoke_model.tokenizer)
    smoke_model.decoder.load_state_dict(jtext_checkpoint['decoder'], strict=True)
    assert smoke_model.encoder.freeze_audio_encoder is True
    assert not any(parameter.requires_grad for parameter in smoke_model.encoder.parameters())
    smoke_model.train()
    assert smoke_model.encoder.audio_enc.training is False
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        smoke_loss = smoke_model(audio.to('cuda'), text)
    smoke_loss.backward()
    assert all(parameter.grad is None for parameter in smoke_model.encoder.parameters())
    assert any(parameter.grad is not None for parameter in smoke_model.decoder.parameters() if parameter.requires_grad)
    print('J-text transfer smoke loss:', float(smoke_loss.detach().cpu()))
finally:
    os.chdir(previous_cwd)
    if smoke_model is not None:
        del smoke_model
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# OnomaCap J-text transfer 학습; train.py의 tqdm이 진행률과 ETA를 표시한다.
assert BEST_JTEXT_PATH.is_file()
os.environ['PYTHONPATH'] = str(captioning_dir)
%cd "{captioning_dir}"
!python train.py --exp_name {EXPERIMENT_NAME} --config settings/onomacap_jamo.yaml --lr 3e-5 --seed {MODEL_SEED}


In [ ]:
# best checkpoint의 full test Jamo Error Rate (추가 지표는 JER만 사용)
from tqdm.auto import tqdm
FOLDER_NAME = f'{EXPERIMENT_NAME}_seed_{MODEL_SEED}'
CHECKPOINT_DIR = Path('/content/drive/MyDrive/OnomaCap/checkpoints') / FOLDER_NAME
BEST_TRANSFER_PATH = CHECKPOINT_DIR / 'best_model.pt'
best_transfer = torch.load(BEST_TRANSFER_PATH, map_location='cpu', weights_only=False)

test_model = BartCaptionModel(transfer_config).to('cuda')
fresh_encoder_state = test_model.encoder.state_dict()
for name, value in fresh_encoder_state.items():
    checkpoint_value = best_transfer['model'][f'encoder.{name}']
    if not torch.equal(value.cpu(), checkpoint_value.cpu()):
        raise RuntimeError(f'Frozen HTSAT changed during transfer: {name}')
test_model.load_state_dict(best_transfer['model'], strict=True)
test_model.eval()
test_loader = AudioCaptionDataModule(transfer_config, 'OnomaCap').test_dataloader()

total_edits = 0
total_reference_tokens = 0
sample_count = 0
with torch.no_grad():
    for batch_data in tqdm(test_loader, desc='OnomaCap test JER', unit='batch'):
        audios, caption_lists, _audio_names, _audio_ids = batch_data
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            predictions = test_model.generate(audios.to('cuda'), num_beams=EVAL_BEAM_SIZE)
        for prediction, references in zip(predictions, caption_lists):
            predicted_tokens = pronounced_hangul_to_jamo(prediction).split()
            reference_token_lists = [reference.split() for reference in references]
            candidates = []
            for reference_tokens in reference_token_lists:
                distance = edit_distance(reference_tokens, predicted_tokens)
                candidates.append((distance / len(reference_tokens), distance, len(reference_tokens)))
            _, selected_distance, selected_length = min(candidates)
            total_edits += selected_distance
            total_reference_tokens += selected_length
            sample_count += 1

test_jer = total_edits / total_reference_tokens
jer_result = {
    'checkpoint': str(BEST_TRANSFER_PATH),
    'epoch': int(best_transfer['epoch']),
    'beam_size': EVAL_BEAM_SIZE,
    'samples': sample_count,
    'total_edits': total_edits,
    'total_reference_tokens': total_reference_tokens,
    'jamo_error_rate': test_jer,
    'reference_policy': 'minimum normalized JER among five references',
}
JER_RESULT_PATH = Path('/content/drive/MyDrive/OnomaCap/results') / FOLDER_NAME / 'test_jer.json'
JER_RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
JER_RESULT_PATH.write_text(json.dumps(jer_result, ensure_ascii=False, indent=2), encoding='utf-8')
display(jer_result)
print('JER result:', JER_RESULT_PATH)
